In [1]:
import pandas as pd
from stock_selection_graph import create_and_compile_graph
from pathlib import Path
import numpy as np
import json
import os
from schemas import StockInfoAndResearch, StockSelectionState
from model_inference import obtain_inference_performance_to_date

In [2]:
with open("api_keys.json", "r", encoding="utf-8") as f:
    api_keys = json.load(f)["api_keys"][0]

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = api_keys["langsmith"]
os.environ["LANGSMITH_PROJECT"] = "portfolio_selection"

In [3]:
inference_results = pd.read_parquet("Data_01_04_2026/results/inference_predictions.parquet")
inference_performance_to_date = obtain_inference_performance_to_date(inference_results)

In [4]:
selected_stocks = [
    "NTSK",
    "INV",
    "MNTN",
    "CHA",
    "AGBK",
    "KLAR",
    "SSII",
    "BETA",
    "WOLF",
    "FLY",
    "EQPT",
    "WYFI",
    "OMDA",
    "TLX",
    "IBTA",
    "FIGR",
    "BBNX",
    "MANE",
    "LMRI",
    "PICS",
    "SAIL",
    "ANTA",
    "XZO",
    "ETOR",
    "PTRN",
    "AERO",
    "HTFL",
    "GLXY",
    "WLTH",
    "BLLN"
]


candidate_investments = inference_results[inference_results.ticker.isin(
    selected_stocks
) & (inference_results.calendardate == "2026-03-31")].sort_values('y_pred')

In [5]:
test_date = '2026-03-31'
TICKER_DATA_PATH = "Data_05_03_2026/SHARADAR_TICKERS.parquet"
ticker_data = pd.read_parquet(TICKER_DATA_PATH)

investments = candidate_investments[candidate_investments.calendardate == test_date]
candidate_investments = candidate_investments[candidate_investments.calendardate == test_date]
stocks_to_consider = candidate_investments.ticker.unique().tolist()

candidate_stocks = inference_results[
    (inference_results.calendardate == test_date ) & (inference_results.ticker.isin(stocks_to_consider))
].copy()

companies = ticker_data[
    ticker_data.isdelisted == "N"
][["ticker", "name", "companysite"]].drop_duplicates()

candidate_stocks = candidate_stocks.merge(companies)
sector_data = ticker_data[['ticker', 'sector']].drop_duplicates().dropna()
candidate_stocks = candidate_stocks.merge(sector_data, on='ticker', how='left')

candidate_stocks = candidate_stocks[[
    "ticker",
    "name",
    "sector",
    "companysite",
    "y_pred"
]].rename(
    columns={
        "y_pred": "prediction_pct_change"
    }
)

In [6]:
def get_input_state(
    candidate_stocks: pd.DataFrame,
    number_of_stocks_to_select: int = 20,
    back_test_date: str = None,
) -> StockSelectionState:
    candidate_stocks_clean = candidate_stocks.astype(object).where(
        pd.notna(candidate_stocks),
        None
    )
    candidate_stocks_dict = candidate_stocks_clean.to_dict(orient="records")
    candidate_stocks = [StockInfoAndResearch(**x) for x in candidate_stocks_dict]

    input_state = StockSelectionState(
        stocks=candidate_stocks,
        back_test_date=back_test_date,
        number_of_stocks_to_select=number_of_stocks_to_select
    )

    return input_state

input_state = get_input_state(candidate_stocks, 20) 

In [8]:
graph = create_and_compile_graph()
response = graph.invoke(input_state)

In [9]:
selected_investments = candidate_investments[candidate_investments.ticker.isin([x.ticker for x in response['portfolio_selection'].selected_stocks])]
deselected_investments = candidate_investments[candidate_investments.ticker.isin([x.ticker for x in response['portfolio_selection'].excluded_stocks])]

In [11]:
response

{'stocks': [StockInfoAndResearch(ticker='AERO', name='GRUPO AEROMEXICO SAB DE CV', sector='Industrials', companysite='https://www.aeromexico.com/es-mx', prediction_pct_change=0.7239158749580383, research=None),
  StockInfoAndResearch(ticker='AGBK', name='AGI INC', sector='Financial Services', companysite='https://agibank.com.br', prediction_pct_change=1.233537197113037, research=None),
  StockInfoAndResearch(ticker='ANTA', name='ANTALPHA PLATFORM HOLDING CO', sector='Financial Services', companysite='https://www.antalpha.com', prediction_pct_change=0.7326064705848694, research=None),
  StockInfoAndResearch(ticker='BBNX', name='BETA BIONICS INC', sector='Healthcare', companysite='https://www.betabionics.com', prediction_pct_change=0.8297722935676575, research=None),
  StockInfoAndResearch(ticker='BETA', name='BETA TECHNOLOGIES INC', sector='Industrials', companysite='https://beta.team', prediction_pct_change=1.0719736814498901, research=None),
  StockInfoAndResearch(ticker='BLLN', name=

In [12]:
import json
from pathlib import Path
from pydantic import BaseModel

def _default(obj):
    if isinstance(obj, BaseModel):
        return obj.model_dump()
    raise TypeError(f"Not serializable: {type(obj)}")

out_path = Path("Data_01_04_2026/results/graph_response.json")
out_path.parent.mkdir(parents=True, exist_ok=True)
out_path.write_text(json.dumps(response, default=_default, indent=2))

845893

In [15]:
candidate_stocks

,ticker,name,sector,companysite,prediction_pct_change
0,AERO,GRUPO AEROMEXICO SAB DE CV,Industrials,https://www.aeromexico.com/es-mx,0.723916
1,AGBK,AGI INC,Financial Services,https://agibank.com.br,1.233537
2,ANTA,ANTALPHA PLATFORM HOLDING CO,Financial Services,https://www.antalpha.com,0.732606
3,BBNX,BETA BIONICS INC,Healthcare,https://www.betabionics.com,0.829772
4,BETA,BETA TECHNOLOGIES INC,Industrials,https://beta.team,1.071974
5,BLLN,BILLIONTOONE INC,Healthcare,https://www.billiontoone.com,0.688935
6,CHA,CHAGEE HOLDINGS LTD,Consumer Cyclical,https://chagee.com/en,1.285712
7,EQPT,EQUIPMENTSHARECOM INC,Industrials,https://www.equipmentshare.com,0.999986
8,ETOR,ETORO GROUP LTD,Financial Services,https://www.etoro.com,0.729404
9,FIGR,FIGURE TECHNOLOGY SOLUTIONS INC,Financial Services,https://www.figure.com,0.846174


In [14]:
selected_investments

,ticker,calendardate,y_pred,close,close_max,marketcap,marketcap_quantile
4657,BLLN,2026-03-31,0.688935,78.94,130.18,3.096257e+09,176145803.0
7063,GLXY,2026-03-31,0.709559,18.45,42.86,8.536775e+09,176145803.0
7776,HTFL,2026-03-31,0.720080,24.33,39.91,2.184250e+09,176145803.0
12027,PTRN,2026-03-31,0.727942,12.43,18.40,1.911071e+09,176145803.0
6050,ETOR,2026-03-31,0.729404,30.03,75.97,2.541053e+09,176145803.0
12782,SAIL,2026-03-31,0.737489,13.24,25.70,6.957334e+09,176145803.0
11619,PICS,2026-03-31,0.743892,10.45,18.00,1.590039e+09,176145803.0
9231,LMRI,2026-03-31,0.777959,8.60,18.53,7.659961e+08,176145803.0
9525,MANE,2026-03-31,0.806563,63.15,67.55,2.272157e+09,176145803.0
7933,IBTA,2026-03-31,0.870209,29.97,109.90,6.416272e+08,176145803.0


In [8]:
comp = pd.concat(
    [
        selected_investments.sort_values('pct_change').pct_change_at_max_date.describe(),
        investments.pct_change_at_max_date.describe()
    ], axis=1
)

comp.columns = ['AI Filtered', 'Normal']
comp

,AI Filtered,Normal
count,20.000000,20.000000
mean,1.363858,2.092430
std,2.653755,3.998254
min,-0.359316,-0.702194
25%,0.331210,0.213362
50%,0.815797,0.793068
75%,1.601507,1.680216
max,12.209702,14.264052


In [39]:
inference_performance_to_date[
    (inference_performance_to_date.ticker.isin(deselected_investments.ticker))
    & (inference_performance_to_date.calendardate == test_date)
].pct_change_at_max_date.describe()

count    10.000000
mean      2.167170
std       4.495558
min      -0.702194
25%       0.164174
50%       0.458805
75%       1.218945
max      14.264052
Name: pct_change_at_max_date, dtype: float64

In [19]:
selected_investments.sort_values('pct_change')

,ticker,calendardate,pct_change,pct_change_at_max_date,close,close_max,marketcap,marketcap_quantile
7734,QBTS,2025-06-30,0.399155,0.291667,14.64,19.04,3.044633e+09,129365753.5
4603,GROY,2025-06-30,0.404873,0.995475,2.21,6.66,2.642553e+08,129365753.5
854,AENT,2025-06-30,0.437870,0.758621,3.77,10.50,2.862700e+08,129365753.5
10411,URG,2025-06-30,0.448137,0.495238,1.05,3.27,2.491716e+08,129365753.5
7938,RBLX,2025-06-30,0.452137,-0.359316,105.20,134.72,4.680162e+10,129365753.5
4503,GPCR,2025-06-30,0.480930,2.043877,20.74,74.30,1.543120e+09,129365753.5
3324,ECG,2025-06-30,0.496731,0.872973,63.53,76.76,3.123214e+09,129365753.5
816,ADUR,2025-06-30,0.604605,0.251116,8.96,10.74,1.312472e+08,129365753.5
3862,EU,2025-06-30,0.620103,-0.262238,2.86,4.94,3.278199e+08,129365753.5
5720,KMTS,2025-06-30,0.633496,0.344391,16.58,25.70,1.258042e+09,129365753.5


In [17]:
(selected_investments['pct_change_at_max_date'] < 0).sum()

np.int64(3)

In [18]:
(investments['pct_change_at_max_date'] < 0).sum()

np.int64(4)

In [13]:
investments[investments.ticker != 'ABVX'].pct_change_at_max_date.describe()

count    19.000000
mean      1.451818
std       2.865397
min      -0.702194
25%       0.175608
50%       0.623064
75%       1.622539
max      12.209702
Name: pct_change_at_max_date, dtype: float64